
## Оглавление
- # 1. Исходные данные и постановка задачи
- # 2. Выполнение задания
  - ## 2.1 Разведочный анализ и отбраковка
  - ## 2.2 Группы полей
  - ## 2.3 Кластерный анализ по сочетаниям признаков
  - ## 2.4 Реализация и пользовательский интерфейс
- # 3. Сравнение вариантов кластеризации и ранжирование
- # 4. Итоговые выводы и выбранное решение
- # 5. Приложение: параметры, версии, воспроизводимость



# 1. Исходные данные и постановка задачи
Описание: выборка пожаров РФ (xls/xlsx) с несколькими листами (склеенный факт-лист и части 2000–2008, 2009–2020, плюс справочники кодов).
Цель: выполнить EDA с чисткой; выделить группы признаков; провести кластеризацию по разным наборам; реализовать интерфейс; сравнить варианты и ранжировать их по близости к ранговой шкале пожаров.


In [ ]:

# 0. Инициализация и версии
import sys, platform, json, random, re
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    normalized_mutual_info_score
)
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

VERSIONS = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sys.modules['sklearn'].__version__,
    'ipywidgets': widgets.__version__,
}
print('Версии:', VERSIONS)

DATA_PATH = next(Path('.').glob('*.xlsx'))
stat = DATA_PATH.stat()
print(f"Файл данных: {DATA_PATH.name}, размер {stat.st_size/1e6:.2f} МБ, изменен {datetime.fromtimestamp(stat.st_mtime)}")


: 

In [2]:

# 1.2 Обзор листов книги
xl = pd.ExcelFile(DATA_PATH)
SHEET_NAMES = xl.sheet_names

sheet_info = []
for name in SHEET_NAMES:
    try:
        df_tmp = xl.parse(name)
        rows, cols = df_tmp.shape
    except Exception:
        rows, cols = None, None
    sheet_info.append({'sheet': name, 'rows': rows, 'cols': cols})

sheet_info_df = pd.DataFrame(sheet_info)
display(sheet_info_df)
FACT_SHEETS = [s for s in SHEET_NAMES if s.startswith(('БД-1','1...','2...'))]
print('Факт-листы для анализа:', FACT_SHEETS)


,sheet,rows,cols
0,БД-1...2000--2020 (1+2),3234,38
1,Исходные 2 части -->,0,0
2,1...2000-2008,2335,47
3,2...2009-2020,902,38
4,Шапки 1 и 2,38,10
5,Табл-Классификаторы -->,0,0
6,Табл.1,87,3
7,Табл.2,11,3
8,Табл.3,10,3
9,Табл.4.1,10,3


Факт-листы для анализа: ['БД-1...2000--2020 (1+2)', '1...2000-2008', '2...2009-2020']



# 2. Выполнение задания
## 2.1 Разведочный анализ и отбраковка
### План контроля качества данных
1. **Дубликаты**: совпадение по всем исходным полям (`RU_COLS`). Действие: флаг `dup_flag`, исключаем из анализа.
2. **Даты**: допустимый интервал 2000–2020. Флаг `flag_date_outlier`, даты вне интервала очищаем; такие строки исключаем из анализа.
3. **Времена событий**: формат часы/минуты, диапазон 0–1439 минут. Флаги `*_invalid`, сводный `flag_time_invalid`. В экспериментах с временными признаками исключаем строки с `flag_time_invalid`.
4. **Числовые поля**: `building_floors>=1 (<=150)`, `fire_floor>=1`, `fire_floor<=building_floors`, `distance_to_station>0 (<=1000)`, потери и ущерб неотрицательны. Флаги `flag_floor_inconsistent`, `flag_negative_values`, `flag_damage_outlier`.
5. **Поля-коды**: извлечение кода (`*_code`) из текстов; если код не извлечён, значения остаются `NaN` и учитываются в отчёте пропусков. Выделяем флаг `flag_missing_outputs` если нет ни одного из выходов.


In [3]:

# 2.1.1 Хелперы загрузки и нормализации
RU_COLS = [
    'N п/п','Субъекты РФ','Дата возникновения пожара','Вид нас. Пункта','Вид пож. Охраны',
    'Категория риска объекта пожара','Тип предприятия','Класс ФПО','Этажность здания',
    'Этаж на котором возник пожар','Степень огнестойкости','Изделие, устройство',
    'Расстояние до пожарной части','Погибло людей: Всего','Получили травмы: Всего',
    'Прямой ущерб','Спасено на пожаре людей','Эвакуировано на пожаре людей',
    'Материальных ценностей','Время обнаружения','Время сообщения',
    'Время прибытия 1-го пож.подразд-ния','Время подачи 1-го ствола','Время локализации',
    'Время ликвидации','Время ликвидации посл. пожара, час. мин.','Техника','Количество техники',
    'Виды стволов','Количество стволов','Огнетушащие средства','Первичные средства',
    'Использование СИЗОД','Водоисточники','Вид АУП','Наименование объекта','Почтовый адрес'
]
EN_COLS = [
    'row_id','region','fire_date','settlement_type','fire_protection','risk_category',
    'enterprise_type','fpo_class','building_floors','fire_floor','fire_resistance','source_item',
    'distance_to_station','fatalities','injuries','direct_damage','people_saved','people_evacuated',
    'assets_saved','t_detect','t_report','t_arrival','t_first_hose','t_contained','t_extinguished',
    't_final_extinguish','equipment','equipment_count','nozzle_types','nozzle_count',
    'extinguishing_agents','initial_means','respirators_use','water_sources','alarm_type',
    'object_name','address'
]
RU_TO_EN = dict(zip(RU_COLS, EN_COLS))
TIME_COLS = ['t_detect','t_report','t_arrival','t_first_hose','t_contained','t_extinguished','t_final_extinguish']

MAX_FLOORS = 150
MAX_DISTANCE = 1000


def norm_col(s):
    s = str(s).replace('\ufffd', ' ')
    s = re.sub(r'\s+', ' ', s.strip())
    return s


def map_col(col: str):
    c = norm_col(col).lower().replace('ё', 'е')
    if c in ('n п/п', 'n п/п.'):
        return 'N п/п'
    if 'код региона' in c or 'субъект' in c:
        return 'Субъекты РФ'
    if c.startswith('дата возник'):
        return 'Дата возникновения пожара'
    if 'вид нас' in c:
        return 'Вид нас. Пункта'
    if 'вид пож' in c or 'пож. охран' in c:
        return 'Вид пож. Охраны'
    if 'категория риска' in c:
        return 'Категория риска объекта пожара'
    if 'тип предпр' in c:
        return 'Тип предприятия'
    if 'класс фпо' in c:
        return 'Класс ФПО'
    if 'этажност' in c:
        return 'Этажность здания'
    if 'этаж' in c and 'возник' in c:
        return 'Этаж на котором возник пожар'
    if 'огнестойк' in c:
        return 'Степень огнестойкости'
    if 'изделие' in c:
        return 'Изделие, устройство'
    if ('расст' in c and 'част' in c) or 'пож.-спас' in c:
        return 'Расстояние до пожарной части'
    if 'погибло' in c:
        return 'Погибло людей: Всего'
    if 'травмы' in c:
        return 'Получили травмы: Всего'
    if 'прямой ущерб' in c:
        return 'Прямой ущерб'
    if 'спасено' in c and 'люд' in c:
        return 'Спасено на пожаре людей'
    if 'эвакуир' in c:
        return 'Эвакуировано на пожаре людей'
    if 'материальных ценност' in c:
        return 'Материальных ценностей'
    if 'обнаруж' in c:
        return 'Время обнаружения'
    if 'сообщ' in c:
        return 'Время сообщения'
    if 'прибытия' in c:
        return 'Время прибытия 1-го пож.подразд-ния'
    if 'подачи' in c and 'ствол' in c:
        return 'Время подачи 1-го ствола'
    if 'локализ' in c:
        return 'Время локализации'
    if 'ликвидации' in c and 'посл' in c:
        return 'Время ликвидации посл. пожара, час. мин.'
    if 'ликвидации' in c:
        return 'Время ликвидации'
    if c.startswith('техник'):
        return 'Техника'
    if 'количество техник' in c or (c.startswith('количество') and 'ствол' not in c):
        return 'Количество техники'
    if 'ствол' in c and 'количество' not in c:
        return 'Виды стволов'
    if 'количество ствол' in c or 'количество .1' in c:
        return 'Количество стволов'
    if 'огнетуш' in c:
        return 'Огнетушащие средства'
    if 'первич' in c:
        return 'Первичные средства'
    if 'сизод' in c:
        return 'Использование СИЗОД'
    if 'водоснабж' in c or 'водоисточ' in c:
        return 'Водоисточники'
    if 'ауп' in c or 'пожарной автоматик' in c:
        return 'Вид АУП'
    if 'наименован' in c:
        return 'Наименование объекта'
    if 'почтовый адрес' in c:
        return 'Почтовый адрес'
    return None


def load_fact_sheet(sheet_name: str, header_row: int | None = None) -> pd.DataFrame:
    header = 3 if sheet_name.startswith('БД-1') else 1
    if header_row is not None:
        header = header_row
    df = xl.parse(sheet_name, header=header)
    df.columns = [norm_col(c) for c in df.columns]
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    col_map = {c: map_col(c) for c in df.columns}
    col_map = {k: v for k, v in col_map.items() if v}
    df = df.rename(columns=col_map)
    df = df.loc[:, ~df.columns.duplicated()]
    for col in RU_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df = df[RU_COLS]
    df['source_sheet'] = sheet_name
    return df


def sheet_period(name: str) -> str:
    if name.startswith('1...'):
        return '2000-2008'
    if name.startswith('2...'):
        return '2009-2020'
    if name.startswith('БД-1'):
        return '2000-2020 (склеено)'
    return 'other'


def first_int(val):
    if pd.isna(val):
        return np.nan
    nums = re.findall(r'\d+', str(val))
    return float(nums[0]) if nums else np.nan


def parse_time(val):
    if pd.isna(val):
        return (np.nan, False)
    s = str(val).strip()
    m = re.search(r'(?P<h>\d{1,2})\D+(?P<m>\d{1,2})$', s)
    if m:
        h = int(m.group('h')); mi = int(m.group('m'))
        invalid = h >= 24 or mi >= 60
        return ((np.nan if invalid else h * 60 + mi), invalid)
    if re.fullmatch(r'\d{1,2}', s):
        h = int(s); invalid = h >= 24
        return ((np.nan if invalid else h * 60), invalid)
    dt = pd.to_datetime(s, errors='coerce')
    if pd.notna(dt):
        h, mi = dt.hour, dt.minute
        invalid = dt.year not in (1899, 1900)
        invalid = invalid or h >= 24 or mi >= 60
        return ((np.nan if invalid else h * 60 + mi), invalid)
    return (np.nan, True)


def normalize_text(series):
    s = series.astype(str).str.strip().str.lower()
    s = s.where(series.notna(), np.nan)
    return s


def compute_rank_ref_v2(severity_series: pd.Series):
    s = severity_series.copy()
    rank = pd.Series(np.nan, index=s.index)
    zero_mask = s.fillna(np.nan).eq(0)
    pos = s[s > 0].dropna()

    rank.loc[zero_mask] = 1

    if len(pos) > 0:
        q = pos.quantile([0.25, 0.5, 0.75]).to_dict()

        def assign_pos(val):
            if val <= q[0.25]:
                return 2
            if val <= q[0.5]:
                return 3
            if val <= q[0.75]:
                return 4
            return 5

        rank.loc[pos.index] = pos.apply(assign_pos)
        return rank, q

    return rank, {}


def clean_fire_data(df: pd.DataFrame):
    df = df.copy()
    df = df.rename(columns=RU_TO_EN)
    df = df.loc[:, ~df.columns.duplicated()]

    df['fire_date'] = pd.to_datetime(df['fire_date'], errors='coerce')
    df['year'] = df['fire_date'].dt.year
    df['month'] = df['fire_date'].dt.month
    df['flag_date_outlier'] = (df['year'] < 2000) | (df['year'] > 2020)
    df.loc[df['flag_date_outlier'], ['fire_date','year','month']] = [pd.NaT, np.nan, np.nan]

    code_cols = [
        'region','settlement_type','fire_protection','risk_category','enterprise_type',
        'fpo_class','fire_resistance','source_item','equipment','nozzle_types',
        'extinguishing_agents','initial_means','respirators_use','water_sources','alarm_type'
    ]
    for col in code_cols:
        df[f'{col}_code'] = df[col].apply(first_int)
        df[f'{col}_text'] = normalize_text(df[col])

    for col in ['equipment_count','nozzle_count']:
        df[col] = df[col].apply(first_int)

    num_cols = [
        'building_floors','fire_floor','distance_to_station','fatalities','injuries',
        'direct_damage','people_saved','people_evacuated','assets_saved',
        'equipment_count','nozzle_count'
    ]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in TIME_COLS:
        parsed = df[col].apply(parse_time)
        df[col + '_min'] = parsed.apply(lambda x: x[0])
        df[col + '_invalid'] = parsed.apply(lambda x: x[1])

    df['direct_damage_log'] = np.log1p(df['direct_damage'])
    severity = (
        df['fatalities'].fillna(0) * 5
        + df['injuries'].fillna(0) * 2
        + df['direct_damage_log']
    )
    all_missing = df[['fatalities', 'injuries', 'direct_damage']].isna().all(axis=1)
    df['severity_score'] = severity.mask(all_missing, np.nan)
    df['rank_ref'], rank_info = compute_rank_ref_v2(df['severity_score'])

    df['flag_floor_outlier'] = df['fire_floor'] > MAX_FLOORS
    df['flag_distance_outlier'] = df['distance_to_station'] > MAX_DISTANCE
    df['flag_floor_inconsistent'] = (df['fire_floor'].notna() & df['building_floors'].notna() & (df['fire_floor'] > df['building_floors']))
    df['flag_negative_values'] = df[num_cols].lt(0).any(axis=1)
    damage_limit = df['direct_damage'].dropna().quantile(0.995) if df['direct_damage'].notna().any() else np.nan
    df['flag_damage_outlier'] = df['direct_damage'] > damage_limit
    df['flag_time_invalid'] = df[[c for c in df.columns if c.endswith('_invalid')]].any(axis=1)
    df['flag_missing_outputs'] = df[['fatalities','injuries','direct_damage']].isna().all(axis=1)

    issues = {
        'invalid_time_counts': {c: int(df[c].sum()) for c in df.columns if c.endswith('_invalid')},
        'floor_outliers': int(df['flag_floor_outlier'].sum()),
        'distance_outliers': int(df['flag_distance_outlier'].sum()),
        'floor_inconsistent': int(df['flag_floor_inconsistent'].sum()),
        'negative_values': int(df['flag_negative_values'].sum()),
        'damage_outliers': int(df['flag_damage_outlier'].sum()),
        'missing_outputs': int(df['flag_missing_outputs'].sum()),
        'rank_quantiles': rank_info,
    }
    return df, issues


In [4]:

# 2.1.2 Формирование raw_df
raw_parts = [load_fact_sheet(s) for s in FACT_SHEETS]
raw_df = pd.concat(raw_parts, ignore_index=True)
raw_df['source_period'] = raw_df['source_sheet'].apply(sheet_period)
raw_df['dup_flag'] = raw_df.duplicated(subset=RU_COLS, keep='first')

print('Всего строк до дедупликации:', len(raw_df))
print('Число дублей:', int(raw_df['dup_flag'].sum()))
raw_df_nodup = raw_df[~raw_df['dup_flag']].copy()
print('Всего строк после дедупликации:', len(raw_df_nodup))
print('Распределение по листам:')
print(raw_df_nodup['source_sheet'].value_counts())
print('Распределение по периодам:')
print(raw_df_nodup['source_period'].value_counts())


Всего строк до дедупликации: 6466
Число дублей: 126
Всего строк после дедупликации: 6340
Распределение по листам:
source_sheet
БД-1...2000--2020 (1+2)    3231
1...2000-2008              2208
2...2009-2020               901
Name: count, dtype: int64
Распределение по периодам:
source_period
2000-2020 (склеено)    3231
2000-2008              2208
2009-2020               901
Name: count, dtype: int64


C:\Users\Maste\AppData\Local\Temp\ipykernel_460164\784355255.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  raw_df = pd.concat(raw_parts, ignore_index=True)


In [5]:

# 2.1.3 Чистка и флаги качества
clean_df, data_quality = clean_fire_data(raw_df_nodup)
print('Форма после чистки:', clean_df.shape)
print('Диапазон лет:', clean_df['year'].min(), '-', clean_df['year'].max())
print('Паспорт качества (флаги):')
flags = [c for c in clean_df.columns if c.startswith('flag_')]
flag_counts = {f: int(clean_df[f].sum()) for f in flags}
print(flag_counts)
print('Суммарная доля проблемных строк:', round(clean_df[flags].any(axis=1).mean()*100,2), '%')
print('Проблемные значения (детально):', data_quality)


Форма после чистки: (6340, 97)
Диапазон лет: 2000.0 - 2020.0
Паспорт качества (флаги):
{'flag_date_outlier': 1, 'flag_floor_outlier': 32, 'flag_distance_outlier': 0, 'flag_floor_inconsistent': 105, 'flag_negative_values': 0, 'flag_damage_outlier': 11, 'flag_time_invalid': 946, 'flag_missing_outputs': 3932}
Суммарная доля проблемных строк: 70.03 %
Проблемные значения (детально): {'invalid_time_counts': {'t_detect_invalid': 534, 't_report_invalid': 456, 't_arrival_invalid': 556, 't_first_hose_invalid': 0, 't_contained_invalid': 632, 't_extinguished_invalid': 501, 't_final_extinguish_invalid': 2, 'flag_time_invalid': 946}, 'floor_outliers': 32, 'distance_outliers': 0, 'floor_inconsistent': 105, 'negative_values': 0, 'damage_outliers': 11, 'missing_outputs': 3932, 'rank_quantiles': {0.25: 7.937731775260109, 0.5: 9.6922415200391, 0.75: 11.454162234689184}}


In [6]:

# 2.1.4 EDA: пропуски, статистики, частоты, распределения

def missing_report(df, top_n=20):
    return (df.isna().mean().sort_values(ascending=False).head(top_n) * 100).round(1).rename('missing_%')

def numeric_summary(df, cols):
    return df[cols].describe().T[['count','mean','std','min','25%','50%','75%','max']]

def freq_top(df, col, n=10):
    return df[col].value_counts(dropna=False).head(n)

print('Top пропуски (20):')
print(missing_report(clean_df))

num_cols_report = ['building_floors','fire_floor','distance_to_station','fatalities','injuries','direct_damage']
print('\nОписательные статистики:')
print(numeric_summary(clean_df, num_cols_report))

for col in ['direct_damage','fire_floor','distance_to_station']:
    if clean_df[col].notna().any():
        idx = clean_df[col].idxmax()
        print(f"\nМаксимум для {col}: {clean_df[col].max()} (строка {idx})")
        display(clean_df.loc[[idx], ['region','fire_date','object_name', col, 'source_sheet']])

print('\nТоп регионов (код):')
print(freq_top(clean_df, 'region_code'))
print('\nТоп типов предприятий (код):')
print(freq_top(clean_df, 'enterprise_type_code'))
print('\nТоп типов населённых пунктов (код):')
print(freq_top(clean_df, 'settlement_type_code'))
print('\nРаспределение по годам:')
print(clean_df['year'].value_counts().sort_index())
print('\nРаспределение по месяцам:')
print(clean_df['month'].value_counts().sort_index())


Top пропуски (20):
people_evacuated          99.5
people_saved              97.5
fatalities                96.6
injuries                  95.2
fpo_class                 92.0
fpo_class_text            92.0
fpo_class_code            92.0
respirators_use           91.6
respirators_use_code      91.6
respirators_use_text      91.6
risk_category             91.1
risk_category_code        91.1
risk_category_text        91.1
nozzle_count              85.8
t_final_extinguish_min    71.6
t_final_extinguish        71.6
t_first_hose              71.6
t_first_hose_min          71.6
alarm_type_code           68.8
address                   67.4
Name: missing_%, dtype: float64

Описательные статистики:
                      count          mean           std  min     25%      50%  \
building_floors      6085.0  1.527855e+00  3.299795e+00  1.0     1.0      1.0   
fire_floor           6068.0  7.120303e+00  6.619976e+01  1.0     1.0      1.0   
distance_to_station  6200.0  8.005968e+00  2.700480e+01  1.0

,region,fire_date,object_name,direct_damage,source_sheet
5564,NaN,NaT,NaN,2.086529e+09,1...2000-2008



Максимум для fire_floor: 908.0 (строка 2915)


,region,fire_date,object_name,fire_floor,source_sheet
2915,1105 (Приморский край),2016-08-06,"крыша цеха № 6 ПАО ВП ""Эра""",908.0,БД-1...2000--2020 (1+2)



Максимум для distance_to_station: 602.0 (строка 771)


,region,fire_date,object_name,distance_to_station,source_sheet
771,1198 (Республика Саха (Якутия)),2001-01-05,энеpгоцех,602.0,БД-1...2000--2020 (1+2)



Топ регионов (код):
region_code
171.0     599
1132.0    527
1104.0    279
1198.0    270
1105.0    204
1125.0    195
1187.0    191
1108.0    190
172.0     183
1157.0    157
Name: count, dtype: int64

Топ типов предприятий (код):
enterprise_type_code
11.0    3527
20.0    2150
27.0     659
NaN        3
12.0       1
Name: count, dtype: int64

Топ типов населённых пунктов (код):
settlement_type_code
1.0    4062
3.0     906
2.0     664
6.0     608
7.0      75
4.0      16
NaN       5
5.0       4
Name: count, dtype: int64

Распределение по годам:
year
2000.0    766
2001.0    773
2002.0    634
2003.0    544
2004.0    544
2005.0    373
2006.0    373
2007.0    271
2008.0    256
2009.0    228
2010.0    202
2011.0    162
2012.0    154
2013.0    134
2014.0    104
2015.0    130
2016.0    126
2017.0    124
2018.0    120
2019.0    180
2020.0    138
Name: count, dtype: int64

Распределение по месяцам:
month
1.0     649
2.0     592
3.0     507
4.0     514
5.0     557
6.0     473
7.0     425
8.0     432


In [7]:

# 2.1.5 Визуализации EDA
fig1 = px.histogram(clean_df, x='direct_damage', nbins=50, title='Прямой ущерб')
fig2 = px.histogram(clean_df, x='direct_damage_log', nbins=50, title='log1p(Прямой ущерб)')
fig3 = px.histogram(clean_df, x='distance_to_station', nbins=50, title='Расстояние до части')
fig4 = px.histogram(clean_df, x='building_floors', nbins=30, title='Этажность здания')
fig5 = px.histogram(clean_df, x='fire_floor', nbins=30, title='Этаж возгорания')

clean_df['period_group'] = np.where(clean_df['year'] <= 2008, '2000-2008', '2009-2020')
fig6 = px.box(clean_df, x='period_group', y='direct_damage_log', title='log1p(ущерб) по периодам')
fig7 = px.box(clean_df, x='settlement_type_code', y='distance_to_station', title='Дистанция до части по типу населённого пункта')

miss = missing_report(clean_df).reset_index().rename(columns={'index':'column'})
fig8 = px.bar(miss, x='column', y='missing_%', title='Top-20 пропусков')
fig8.update_layout(xaxis_tickangle=-60)

for fig in [fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8]:
    fig.show()



### Правила отбраковки для анализа
Исключаем строки:
- `dup_flag == True`;
- `flag_date_outlier == True`;
- `flag_negative_values == True`;
- `flag_floor_inconsistent == True`;
- в экспериментах с временными признаками дополнительно исключаются строки с `flag_time_invalid == True`.
Дополнительно фиксируем флаги, но не удаляем: `flag_damage_outlier`, `flag_missing_outputs`, `flag_distance_outlier`, `flag_floor_outlier`.


In [8]:

# 2.1.6 Формирование analysis_df
rules = {
    'dup_flag': clean_df['dup_flag'],
    'flag_date_outlier': clean_df['flag_date_outlier'],
    'flag_negative_values': clean_df['flag_negative_values'],
    'flag_floor_inconsistent': clean_df['flag_floor_inconsistent'],
}

current = clean_df.copy()
removed_log = {}
for name, cond in rules.items():
    removed_log[name] = int(cond.loc[current.index].sum())
    current = current.loc[~cond]
analysis_df = current.reset_index(drop=True)

print('Размер analysis_df:', analysis_df.shape)
print('Удалено по правилам:')
for k,v in removed_log.items():
    print(f"- {k}: {v}")
print('Итоговая доля отброшенных:', round((len(clean_df)-len(analysis_df))/len(clean_df)*100,2), '%')


Размер analysis_df: (6235, 98)
Удалено по правилам:
- dup_flag: 0
- flag_date_outlier: 1
- flag_negative_values: 0
- flag_floor_inconsistent: 104
Итоговая доля отброшенных: 1.66 %



## 2.2 Группы полей
**Определения**:
- Управляемые: ресурсы и организация тушения (охрана, риск, техника, стволы, средства, сигнализация, времена реагирования, расстояние).
- Наблюдаемые: контекст и окружение (регион, тип населённого пункта, тип предприятия, класс ФПО, этажность, этаж пожара, сезонность).
- Выходные: последствия (погибшие, травмы, прямой ущерб, интегральный severity_score, rank_ref).


In [9]:

FEATURE_GROUPS = {
    'managed': [
        'fire_protection_code','risk_category_code','equipment_count','nozzle_count',
        'extinguishing_agents_code','initial_means_code','alarm_type_code',
        't_report_min','t_arrival_min','t_first_hose_min','t_contained_min','t_extinguished_min',
        'distance_to_station'
    ],
    'observed': [
        'region_code','settlement_type_code','enterprise_type_code','fpo_class_code',
        'building_floors','fire_floor','year','month','distance_to_station'
    ],
    'outputs': ['fatalities','injuries','direct_damage_log','severity_score'],
    'mixed_core': [
        'region_code','enterprise_type_code','fire_protection_code','risk_category_code',
        'building_floors','fire_floor','distance_to_station','equipment_count','nozzle_count',
        'direct_damage_log'
    ],
    'managed_observed': [
        'fire_protection_code','risk_category_code','equipment_count','nozzle_count','extinguishing_agents_code',
        'initial_means_code','alarm_type_code','t_report_min','t_arrival_min','t_first_hose_min','t_contained_min','t_extinguished_min',
        'region_code','settlement_type_code','enterprise_type_code','fpo_class_code','building_floors','fire_floor','year','month','distance_to_station'
    ],
    'observed_outputs': [
        'region_code','settlement_type_code','enterprise_type_code','fpo_class_code','building_floors','fire_floor','year','month','distance_to_station',
        'direct_damage_log','severity_score'
    ],
    'managed_outputs': [
        'fire_protection_code','risk_category_code','equipment_count','nozzle_count','extinguishing_agents_code','initial_means_code','alarm_type_code',
        't_report_min','t_arrival_min','t_first_hose_min','t_contained_min','t_extinguished_min','distance_to_station',
        'direct_damage_log','severity_score'
    ],
}

MISSING_THRESHOLD = 0.9
for name, feats in FEATURE_GROUPS.items():
    missing_rates = analysis_df[feats].isna().mean()
    print(f"\nГруппа {name} ({len(feats)} признаков):")
    print(missing_rates.round(3))
    too_missing = missing_rates[missing_rates > MISSING_THRESHOLD].index.tolist()
    if too_missing:
        print('Исключить из экспериментов (пропуски>90%):', too_missing)

def filter_features_by_missing(df, features, thr=0.9):
    rates = df[features].isna().mean()
    return [f for f in features if rates[f] <= thr]

FEATURE_GROUPS_USED = {}
for name, feats in FEATURE_GROUPS.items():
    filtered = filter_features_by_missing(analysis_df, feats, MISSING_THRESHOLD)
    if len(filtered) >= 2:
        FEATURE_GROUPS_USED[name] = filtered

print('FEATURE_GROUPS_USED:', {k: len(v) for k, v in FEATURE_GROUPS_USED.items()})



Группа managed (13 признаков):
fire_protection_code         0.156
risk_category_code           0.913
equipment_count              0.059
nozzle_count                 0.859
extinguishing_agents_code    0.079
initial_means_code           0.111
alarm_type_code              0.690
t_report_min                 0.073
t_arrival_min                0.089
t_first_hose_min             0.718
t_contained_min              0.099
t_extinguished_min           0.079
distance_to_station          0.022
dtype: float64
Исключить из экспериментов (пропуски>90%): ['risk_category_code']

Группа observed (9 признаков):
region_code             0.000
settlement_type_code    0.001
enterprise_type_code    0.000
fpo_class_code          0.922
building_floors         0.041
fire_floor              0.044
year                    0.000
month                   0.000
distance_to_station     0.022
dtype: float64
Исключить из экспериментов (пропуски>90%): ['fpo_class_code']

Группа outputs (4 признаков):
fatalities           0


## 2.3 Кластерный анализ по сочетаниям признаков
### План экспериментов
- Наборы признаков: managed, observed, outputs, mixed_core, managed+observed, observed+outputs, managed+outputs.
- Обработка пропусков: A) dropna; B) imputation (медиана для числовых, мода для кодов).
- Алгоритмы: KMeans (базовый), AgglomerativeClustering (контроль).
- k: перебор 2..10.
- Масштабирование: StandardScaler.
- Для наборов с временами исключаем строки `flag_time_invalid`.


In [ ]:

# 2.3.1 Функции запуска экспериментов
 

experiments = []
run_history = {}


def impute_values(df, features):
    fills = {}
    for col in features:
        series = df[col]
        if pd.api.types.is_numeric_dtype(series):
            fills[col] = series.median()
        else:
            fills[col] = series.mode().iloc[0] if not series.mode().empty else 0
    return fills


def prepare_features(df, features, imputation='dropna'):
    work = df.copy()
    if any(f.endswith('_min') for f in features):
        work = work[~work['flag_time_invalid']].copy()
    cols_needed = list(dict.fromkeys(features + ['row_id', 'rank_ref', 'severity_score']))
    work = work.loc[:, cols_needed]
    if imputation == 'dropna':
        work = work.dropna(subset=features + ['rank_ref'])
    else:
        fills = impute_values(work, features)
        work = work.copy()
        work[features] = work[features].fillna(fills)
        work = work.dropna(subset=['rank_ref'])
    return work


def compute_alignment_metrics(df_with_labels, label_col='cluster'):
    cluster_means = df_with_labels.groupby(label_col)['rank_ref'].mean()
    if isinstance(cluster_means, pd.DataFrame):
        cluster_means = cluster_means.iloc[:, 0]
    if isinstance(cluster_means, np.ndarray):
        cluster_means = pd.Series(cluster_means.ravel())
    cluster_means = pd.Series(cluster_means).dropna().sort_values()
    if cluster_means.empty:
        return np.nan, np.nan, np.nan, {}
    order_map = {lab: i+1 for i, lab in enumerate(cluster_means.index)}
    mapped = df_with_labels[label_col].map(order_map)
    rank_series = df_with_labels['rank_ref']
    if isinstance(rank_series, pd.DataFrame):
        rank_series = rank_series.iloc[:, 0]
    rank_series = rank_series.dropna()
    mapped_rank = mapped.loc[rank_series.index]
    spearman = spearmanr(mapped_rank, rank_series).statistic if len(rank_series) > 1 else np.nan
    purity_vals = []
    for lab, grp in df_with_labels.groupby(label_col):
        rank_part = grp['rank_ref']
        if isinstance(rank_part, pd.DataFrame):
            rank_part = rank_part.iloc[:, 0]
        counts = rank_part.value_counts(dropna=True)
        purity_vals.append(counts.max() / len(grp) if len(grp) else 0)
    purity = float(np.mean(purity_vals)) if purity_vals else np.nan
    if len(rank_series) > 0:
        rank_int = rank_series.astype(int)
        cl_int = df_with_labels.loc[rank_series.index, label_col].astype(int)
        nmi = normalized_mutual_info_score(rank_int, cl_int)
    else:
        nmi = np.nan
    return spearman, purity, nmi, order_map


def run_experiment(df, feature_set_name, features, imputation, algorithm, k, run_id):
    prepared = prepare_features(df, features, imputation)
    rows_used = len(prepared)
    if rows_used < max(k * 2, 10):
        return None
    X = StandardScaler().fit_transform(prepared[features])
    if algorithm == 'kmeans':
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED)
    else:
        model = AgglomerativeClustering(n_clusters=k)
    labels = model.fit_predict(X)
    prepared = prepared.assign(cluster=labels)

    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    db = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan
    ch = calinski_harabasz_score(X, labels) if len(set(labels)) > 1 else np.nan
    inertia = model.inertia_ if hasattr(model, 'inertia_') else np.nan

    spearman, purity, nmi, order_map = compute_alignment_metrics(prepared)

    cluster_profile = prepared.groupby('cluster').mean()
    size_stats = prepared['cluster'].value_counts()
    pca = PCA(n_components=2, random_state=SEED).fit_transform(X)

    result = {
        'run_id': run_id,
        'feature_set': feature_set_name,
        'features': ','.join(features),
        'imputation': imputation,
        'algorithm': algorithm,
        'k': k,
        'rows_used': rows_used,
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski_harabasz': ch,
        'inertia': inertia,
        'cluster_min': size_stats.min(),
        'cluster_max': size_stats.max(),
        'cluster_smallest_share': size_stats.min()/rows_used,
        'spearman': spearman,
        'purity': purity,
        'nmi': nmi,
    }

    experiments.append(result)
    run_history[run_id] = {
        'params': result,
        'features': features,
        'labels': labels.tolist(),
        'indices': prepared.index.tolist(),
        'row_id_values': prepared['row_id'].tolist() if 'row_id' in prepared.columns else prepared.index.tolist(),
        'cluster_profile': cluster_profile,
        'pca': pca,
        'order_map': order_map,
    }
    return result


In [11]:

# 2.3.2 Массовый прогон экспериментов
experiments = []
run_history = {}
run_id = 0
for name, feats in FEATURE_GROUPS_USED.items():
    for imputation in ['dropna','impute']:
        for algorithm in ['kmeans','agglo']:
            for k in range(2, 11):
                res = run_experiment(analysis_df, name, feats, imputation, algorithm, k, run_id)
                run_id += 1

results_df = pd.DataFrame(experiments)
if not results_df.empty:
    results_df['silhouette_norm'] = results_df['silhouette'].clip(lower=0).fillna(0)
    results_df['db_norm'] = 1 / (1 + results_df['davies_bouldin'].fillna(np.inf))
    results_df['spearman_norm'] = (results_df['spearman'].fillna(0) + 1) / 2
    results_df['purity_norm'] = results_df['purity'].fillna(0)
    results_df['nmi_norm'] = results_df['nmi'].fillna(0)
    results_df['final_score'] = (
        0.3 * results_df['silhouette_norm'] +
        0.2 * results_df['db_norm'] +
        0.2 * results_df['spearman_norm'] +
        0.15 * results_df['purity_norm'] +
        0.15 * results_df['nmi_norm']
    )
    results_df = results_df.loc[:, ~results_df.columns.duplicated()].copy()
    results_df['final_score'] = pd.to_numeric(results_df['final_score'], errors='coerce')
    min_share = 0.01
    min_cluster = 30
    results_df = results_df[(results_df['cluster_smallest_share'] >= min_share) & (results_df['cluster_min'] >= min_cluster)]
    results_df = results_df.sort_values(by=['final_score'], ascending=False)
    display(results_df.head(10))
    results_df.to_csv('experiments_results.csv', index=False)
else:
    print('Нет успешных экспериментов')


,run_id,feature_set,features,imputation,algorithm,k,rows_used,silhouette,davies_bouldin,calinski_harabasz,inertia,cluster_min,cluster_max,cluster_smallest_share,spearman,purity,nmi,silhouette_norm,db_norm,spearman_norm,purity_norm,nmi_norm,final_score
5,23,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,7,1700,0.296683,1.328197,291.729440,10030.038420,30,700,0.017647,0.237754,0.449631,0.037521,0.296683,0.429517,0.618877,0.449631,0.037521,0.371756
3,21,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,5,1700,0.300339,1.307684,329.566329,11475.259015,52,777,0.030588,0.190629,0.451279,0.029144,0.300339,0.433335,0.595315,0.451279,0.029144,0.367895
2,20,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,4,1700,0.295116,1.295703,358.324285,12486.011888,54,829,0.031765,0.164436,0.465353,0.027993,0.295116,0.435596,0.582218,0.465353,0.027993,0.366100
4,22,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,6,1700,0.302706,1.326732,316.046153,10554.417966,52,712,0.030588,0.186162,0.426333,0.028889,0.302706,0.429787,0.593081,0.426333,0.028889,0.363669
36,54,observed,"region_code,settlement_type_code,enterprise_ty...",impute,kmeans,2,2031,0.372092,1.306881,334.848582,13946.405982,244,1787,0.120138,0.103414,0.304756,0.009012,0.372092,0.433486,0.551707,0.304756,0.009012,0.355731
1,19,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,3,1700,0.288762,1.298109,416.718826,13680.953557,54,854,0.031765,0.065555,0.373346,0.008698,0.288762,0.435140,0.532778,0.373346,0.008698,0.337519
144,198,observed_outputs,"region_code,settlement_type_code,enterprise_ty...",impute,kmeans,2,2031,0.330803,1.902199,262.489524,17983.494824,364,1667,0.179222,0.143562,0.315518,0.013551,0.330803,0.344566,0.571781,0.315518,0.013551,0.331871
12,30,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,agglo,5,1700,0.255725,1.553245,275.520929,NaN,50,735,0.029412,0.131566,0.399552,0.022636,0.255725,0.391658,0.565783,0.399552,0.022636,0.331534
11,29,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,agglo,4,1700,0.246120,1.541974,279.875578,NaN,50,780,0.029412,0.133423,0.405734,0.019515,0.246120,0.393395,0.566711,0.405734,0.019515,0.329645
163,235,managed_outputs,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,3,1700,0.260389,1.404375,335.223808,17059.976213,54,850,0.031765,0.071486,0.374989,0.008920,0.260389,0.415908,0.535743,0.374989,0.008920,0.326033


In [12]:

# 2.3.3 Визуализация результатов кластеризации
if not results_df.empty:
    for metric in ['silhouette','davies_bouldin']:
        fig = px.line(results_df, x='k', y=metric, color='feature_set', line_dash='algorithm', symbol='imputation',
                      title=f'{metric} по k')
        fig.show()

    best_per_set = results_df.groupby('feature_set').head(1)
    for _, row in best_per_set.iterrows():
        rh = run_history.get(row['run_id'])
        if rh is None:
            continue
        pca_df = pd.DataFrame(rh['pca'], columns=['pc1','pc2'])
        pca_df['cluster'] = rh['labels']
        fig = px.scatter(pca_df, x='pc1', y='pc2', color='cluster', title=f"PCA: {row['feature_set']} k={row['k']} ({row['algorithm']}/{row['imputation']})")
        fig.show()

    top3 = results_df.head(3)
    for _, row in top3.iterrows():
        rh = run_history.get(row['run_id'])
        if rh is None:
            continue
        print(f"Профиль кластеров: {row['feature_set']} k={row['k']} {row['algorithm']} {row['imputation']}")
        display(rh['cluster_profile'])
        print('Интерпретация: более высокие severity_score/rank_ref соответствуют более тяжёлым кластерам; порядок кластеров:', rh['order_map'])


Профиль кластеров: managed k=7 kmeans impute


,fire_protection_code,equipment_count,nozzle_count,extinguishing_agents_code,initial_means_code,alarm_type_code,t_report_min,t_arrival_min,t_first_hose_min,t_contained_min,t_extinguished_min,distance_to_station,row_id,rank_ref,severity_score
cluster,,,,,,,,,,,,,,,
0,1.270936,2.298851,1.096880,11.172414,19.206897,0.013136,937.285714,988.817734,789.889984,984.331691,979.440066,5.620690,115.494253,3.288998,9.310868
1,1.291429,2.501429,1.037143,11.088571,19.595714,0.020000,101.825000,71.331429,627.104286,99.135714,167.999286,5.150000,127.616595,3.447143,10.803883
2,4.222222,1.811966,0.991453,11.051282,17.615385,0.102564,382.085470,371.598291,658.282051,381.350427,439.111111,22.615385,128.709402,3.512821,10.166248
3,1.274510,3.950980,1.127451,11.764706,17.117647,2.343137,576.529412,539.637255,688.754902,707.696078,672.264706,5.382353,134.803922,3.950980,11.138489
4,1.961538,0.153846,0.500000,0.000000,8.423077,0.038462,502.269231,240.923077,81.923077,248.384615,317.807692,20.576923,41.615385,4.269231,11.320874
5,1.000000,7.133333,5.066667,11.133333,18.266667,0.500000,707.900000,713.500000,718.900000,773.933333,769.233333,3.266667,34.8,4.633333,13.639097
6,1.133333,9.211111,1.100000,11.211111,20.622222,0.177778,665.144444,697.366667,703.811111,764.033333,921.166667,3.877778,95.211111,4.300000,11.782253


Интерпретация: более высокие severity_score/rank_ref соответствуют более тяжёлым кластерам; порядок кластеров: {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 6: 6, 5: 7}
Профиль кластеров: managed k=5 kmeans impute


,fire_protection_code,equipment_count,nozzle_count,extinguishing_agents_code,initial_means_code,alarm_type_code,t_report_min,t_arrival_min,t_first_hose_min,t_contained_min,t_extinguished_min,distance_to_station,row_id,rank_ref,severity_score
cluster,,,,,,,,,,,,,,,
0,1.459344,2.737518,1.041369,11.166904,19.296719,0.011412,909.562054,965.223966,773.539230,964.573466,976.502140,6.336662,117.415121,3.350927,9.456595
1,1.396552,4.474138,1.129310,11.672414,17.275862,2.275862,561.887931,530.413793,681.965517,705.000000,663.189655,7.577586,124.758621,3.948276,11.235126
2,1.944444,0.148148,0.500000,0.407407,8.518519,0.037037,536.259259,232.000000,78.888889,239.185185,321.222222,22.555556,40.851852,4.296296,11.351309
3,1.536680,2.621622,1.029601,11.081081,19.454311,0.020592,99.160232,67.262548,626.268983,94.916345,173.814028,6.396396,127.323454,3.477477,10.806962
4,1.000000,6.423077,4.115385,11.307692,18.384615,0.307692,846.634615,853.788462,850.538462,883.480769,820.692308,4.442308,44.942308,4.557692,12.972049


Интерпретация: более высокие severity_score/rank_ref соответствуют более тяжёлым кластерам; порядок кластеров: {0: 1, 3: 2, 1: 3, 2: 4, 4: 5}
Профиль кластеров: managed k=4 kmeans impute


,fire_protection_code,equipment_count,nozzle_count,extinguishing_agents_code,initial_means_code,alarm_type_code,t_report_min,t_arrival_min,t_first_hose_min,t_contained_min,t_extinguished_min,distance_to_station,row_id,rank_ref,severity_score
cluster,,,,,,,,,,,,,,,
0,1.000000,6.933333,3.850000,11.316667,18.150000,0.800000,839.183333,846.416667,840.100000,890.450000,850.050000,4.316667,43.166667,4.600000,13.191371
1,1.524729,2.683957,1.032569,11.117008,19.408926,0.154403,99.299759,68.414958,622.556092,103.336550,178.992159,6.406514,127.660628,3.517491,10.843374
2,1.462351,2.863937,1.040951,11.208719,19.054161,0.169089,913.056803,960.778071,773.702774,975.619551,977.278732,6.541612,118.397622,3.373844,9.541961
3,1.944444,0.148148,0.500000,0.407407,8.518519,0.037037,536.259259,232.000000,78.888889,239.185185,321.222222,22.555556,40.851852,4.296296,11.351309


Интерпретация: более высокие severity_score/rank_ref соответствуют более тяжёлым кластерам; порядок кластеров: {2: 1, 1: 2, 3: 3, 0: 4}



## 2.4 Реализация и пользовательский интерфейс
Функции интерфейса:
- **EDA**: выбор столбца, просмотр распределения, отчёт по пропускам, фильтры по году/региону, доли флагов качества.
- **Clustering**: выбор признаков, k, алгоритм, стратегия пропусков, запуск; вывод метрик, профилей кластеров, PCA-графика.
- **Compare/Export**: список сохранённых запусков, сравнение метрик, назначение лучшего, экспорт меток и параметров в файлы.


In [ ]:

# 2.4 Интерфейс ipywidgets
all_numeric = [c for c in analysis_df.columns if pd.api.types.is_numeric_dtype(analysis_df[c])]
eda_col = widgets.Dropdown(options=all_numeric, value=(all_numeric[0] if all_numeric else None), description='Колонка')
year_min = int(np.nanmin(analysis_df['year'])) if analysis_df['year'].notna().any() else 2000
year_max = int(np.nanmax(analysis_df['year'])) if analysis_df['year'].notna().any() else 2020
year_slider = widgets.IntRangeSlider(value=[year_min, year_max], min=year_min, max=year_max, step=1, description='Годы')
region_options = ['all'] + sorted([int(x) for x in analysis_df['region_code'].dropna().unique()])
region_dropdown = widgets.Dropdown(options=region_options, value='all', description='Регион')
eda_btn = widgets.Button(description='Показать EDA', button_style='info')
eda_output = widgets.Output()


def render_eda(_):
    with eda_output:
        eda_output.clear_output()
        df = analysis_df.copy()
        if 'year' in df.columns:
            df = df[(df['year'] >= year_slider.value[0]) & (df['year'] <= year_slider.value[1])]
        if region_dropdown.value != 'all':
            df = df[df['region_code'] == region_dropdown.value]
        col = eda_col.value
        print('Строк:', len(df))
        if col:
            miss = df[col].isna().mean() * 100
            print(f'Пропуски {col}: {miss:.1f}%')
            fig = px.histogram(df, x=col, nbins=50, title=f'Распределение {col}')
            fig.show()
        flag_cols = [c for c in df.columns if c.startswith('flag_')]
        if flag_cols:
            flag_rate = df[flag_cols].mean().sort_values(ascending=False).head(10)
            display(flag_rate.rename('flag_rate').to_frame())

eda_btn.on_click(render_eda)

feature_selector = widgets.SelectMultiple(options=all_numeric, value=tuple(FEATURE_GROUPS['outputs']), description='Признаки', rows=12)
clusters_slider = widgets.IntSlider(value=4, min=2, max=10, step=1, description='k')
imputation_dropdown = widgets.Dropdown(options=['dropna','impute'], value='dropna', description='Пропуски')
algorithm_dropdown = widgets.Dropdown(options=['kmeans','agglo'], value='kmeans', description='Алгоритм')
run_btn = widgets.Button(description='Запустить', button_style='success')
run_output = widgets.Output()

history_select = widgets.Dropdown(options=[], description='Запуск')
compare_select1 = widgets.Dropdown(options=[], description='Запуск A')
compare_select2 = widgets.Dropdown(options=[], description='Запуск B')
compare_btn = widgets.Button(description='Сравнить', button_style='warning')
compare_output = widgets.Output()
set_best_btn = widgets.Button(description='Назначить лучшим', button_style='info')
export_btn = widgets.Button(description='Экспорт меток', button_style='warning')
best_variant = {}


def register_history_option(run_id, params):
    label = f"#{run_id} {params['feature_set']} k={params['k']} {params['algorithm']} {params['imputation']} score={params.get('final_score',''):.3f}"
    opts = list(history_select.options)
    opts.append((label, run_id))
    history_select.options = opts
    compare_select1.options = opts
    compare_select2.options = opts


def launch_custom(_):
    with run_output:
        run_output.clear_output()
        feats = list(feature_selector.value)
        res = run_experiment(analysis_df, 'custom', feats, imputation_dropdown.value, algorithm_dropdown.value, clusters_slider.value, max(run_history.keys(), default=0)+1)
        if res is None:
            print('Недостаточно данных для кластеризации')
            return
        res['final_score'] = res['silhouette'] if pd.notna(res['silhouette']) else 0
        print('Метрики запуска:', res)
        register_history_option(res['run_id'], res)

run_btn.on_click(launch_custom)


def on_select_change(change):
    run_id = change['new']
    if run_id is None:
        return
    rh = run_history.get(run_id)
    if rh is None:
        return
    with run_output:
        run_output.clear_output()
        print('Параметры:', rh['params'])
        display(rh['cluster_profile'])
        pca_df = pd.DataFrame(rh['pca'], columns=['pc1','pc2'])
        pca_df['cluster'] = rh['labels']
        fig = px.scatter(pca_df, x='pc1', y='pc2', color='cluster', title=f"PCA запуск {run_id}")
        fig.show()

history_select.observe(on_select_change, names='value')


def compare_runs(_):
    with compare_output:
        compare_output.clear_output()
        a = compare_select1.value
        b = compare_select2.value
        if a is None or b is None or a == b:
            print('Выберите два разных запуска')
            return
        ra = run_history.get(a)
        rb = run_history.get(b)
        if ra is None or rb is None:
            print('Запуски не найдены')
            return
        keys = ['rows_used','silhouette','davies_bouldin','calinski_harabasz','final_score','spearman','purity','nmi','cluster_min','cluster_max','cluster_smallest_share']
        df_cmp = pd.DataFrame({
            'A': [ra['params'].get(k) for k in keys],
            'B': [rb['params'].get(k) for k in keys],
        }, index=keys)
        df_cmp['delta'] = df_cmp['A'] - df_cmp['B']
        display(df_cmp)

        for title, r in [('A', ra), ('B', rb)]:
            pca_df = pd.DataFrame(r['pca'], columns=['pc1','pc2'])
            pca_df['cluster'] = r['labels']
            fig = px.scatter(pca_df, x='pc1', y='pc2', color='cluster', title=f"PCA запуск {title}")
            fig.show()

compare_btn.on_click(compare_runs)


def set_best(_):
    if history_select.value is None:
        return
    best_variant.clear()
    best_variant.update(run_history[history_select.value]['params'])
    print('Лучший вариант обновлен:', best_variant)

set_best_btn.on_click(set_best)


def export_labels(_):
    if history_select.value is None:
        print('Не выбран запуск')
        return
    rh = run_history[history_select.value]
    labels_df = pd.DataFrame({'row_id': rh['row_id_values'], 'cluster': rh['labels']})
    labels_df.to_csv('labels_best.csv', index=False)
    if best_variant:
        with open('best_variant.json','w',encoding='utf-8') as f:
            json.dump(best_variant, f, ensure_ascii=False, indent=2, default=str)
    print('Экспортировано labels_best.csv и best_variant.json (если выбран лучший)')

export_btn.on_click(export_labels)

ui = widgets.Tab()
ui.children = [
    widgets.VBox([eda_col, year_slider, region_dropdown, eda_btn, eda_output]),
    widgets.VBox([feature_selector, clusters_slider, imputation_dropdown, algorithm_dropdown, run_btn, run_output]),
    widgets.VBox([history_select, compare_select1, compare_select2, compare_btn, compare_output, set_best_btn, export_btn, run_output])
]
ui.set_title(0, 'EDA')
ui.set_title(1, 'Clustering')
ui.set_title(2, 'Compare/Export')
display(ui)



# 3. Сравнение вариантов кластеризации и ранжирование
Эталонная линейка `rank_ref`: нулевые значения severity_score выделяются в класс 1, положительные значения разбиваются по квантилям 25/50/75% на классы 2–5. Для каждого варианта считаются согласованность (Spearman по упорядоченным кластерам), однородность (purity по rank_ref), информационная близость (NMI). Итоговый балл `final_score` комбинирует silhouette, Davies-Bouldin (обратная), Spearman, purity, NMI с весами 0.3/0.2/0.2/0.15/0.15.


In [14]:

# 3.1 Ранжирование вариантов и сохранение лучших
if 'results_df' in globals() and not results_df.empty:
    print('Топ-10 по final_score:')
    display(results_df.head(10))
    best_row = results_df.iloc[0]
    best_run_id = int(best_row['run_id'])
    best_art = run_history.get(best_run_id)
    if best_art:
        best_labels_df = pd.DataFrame({
            'row_id': best_art['row_id_values'],
            'cluster_best': best_art['labels'],
            'rank_ref': analysis_df.loc[best_art['indices'], 'rank_ref'].values,
        })
        best_labels_df.to_csv('labels_best.csv', index=False)
        with open('best_variant.json','w',encoding='utf-8') as f:
            json.dump(best_art['params'], f, ensure_ascii=False, indent=2, default=str)
        print('Лучший вариант сохранён в best_variant.json, метки — labels_best.csv')
else:
    print('Нет результатов для ранжирования')


Топ-10 по final_score:


,run_id,feature_set,features,imputation,algorithm,k,rows_used,silhouette,davies_bouldin,calinski_harabasz,inertia,cluster_min,cluster_max,cluster_smallest_share,spearman,purity,nmi,silhouette_norm,db_norm,spearman_norm,purity_norm,nmi_norm,final_score
5,23,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,7,1700,0.296683,1.328197,291.729440,10030.038420,30,700,0.017647,0.237754,0.449631,0.037521,0.296683,0.429517,0.618877,0.449631,0.037521,0.371756
3,21,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,5,1700,0.300339,1.307684,329.566329,11475.259015,52,777,0.030588,0.190629,0.451279,0.029144,0.300339,0.433335,0.595315,0.451279,0.029144,0.367895
2,20,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,4,1700,0.295116,1.295703,358.324285,12486.011888,54,829,0.031765,0.164436,0.465353,0.027993,0.295116,0.435596,0.582218,0.465353,0.027993,0.366100
4,22,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,6,1700,0.302706,1.326732,316.046153,10554.417966,52,712,0.030588,0.186162,0.426333,0.028889,0.302706,0.429787,0.593081,0.426333,0.028889,0.363669
36,54,observed,"region_code,settlement_type_code,enterprise_ty...",impute,kmeans,2,2031,0.372092,1.306881,334.848582,13946.405982,244,1787,0.120138,0.103414,0.304756,0.009012,0.372092,0.433486,0.551707,0.304756,0.009012,0.355731
1,19,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,3,1700,0.288762,1.298109,416.718826,13680.953557,54,854,0.031765,0.065555,0.373346,0.008698,0.288762,0.435140,0.532778,0.373346,0.008698,0.337519
144,198,observed_outputs,"region_code,settlement_type_code,enterprise_ty...",impute,kmeans,2,2031,0.330803,1.902199,262.489524,17983.494824,364,1667,0.179222,0.143562,0.315518,0.013551,0.330803,0.344566,0.571781,0.315518,0.013551,0.331871
12,30,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,agglo,5,1700,0.255725,1.553245,275.520929,NaN,50,735,0.029412,0.131566,0.399552,0.022636,0.255725,0.391658,0.565783,0.399552,0.022636,0.331534
11,29,managed,"fire_protection_code,equipment_count,nozzle_co...",impute,agglo,4,1700,0.246120,1.541974,279.875578,NaN,50,780,0.029412,0.133423,0.405734,0.019515,0.246120,0.393395,0.566711,0.405734,0.019515,0.329645
163,235,managed_outputs,"fire_protection_code,equipment_count,nozzle_co...",impute,kmeans,3,1700,0.260389,1.404375,335.223808,17059.976213,54,850,0.031765,0.071486,0.374989,0.008920,0.260389,0.415908,0.535743,0.374989,0.008920,0.326033


Лучший вариант сохранён в best_variant.json, метки — labels_best.csv


# 4. Итоговые выводы и выбранное решение

## 4.1. Итоговые выводы по данным и качеству

1. Объединённая выборка за 2000–2020 гг. после удаления полных дублей содержит 6340 записей; выявлено 126 дублей. Рабочая выборка для кластеризации после исключения записей с выбросами даты и логически противоречивыми этажами составила 6235 записей (отброшено 1.66%).

2. Данные характеризуются высокой долей пропусков по ряду ключевых полей, особенно по выходным показателям последствий:

   * `fatalities` (погибшие) — заполнено 213 наблюдений (≈ 3.4%),
   * `injuries` (травмы) — заполнено 304 наблюдения (≈ 4.8%),
   * `direct_damage` (прямой ущерб) — заполнено 2068 наблюдений (≈ 32.6%),
   * дополнительные выходы (`people_saved`, `people_evacuated`) заполнены крайне редко (≈ 2.5% и ≈ 0.5% соответственно).
     Это означает, что анализ тяжести пожара и ранжирование по последствиям возможно только на подвыборке, где выходы присутствуют, и требует явного указания ограничений.

3. Временные поля имеют значимый процент некорректных или непарсимых значений: `flag_time_invalid` зафиксирован у 946 записей. Следовательно, признаки временной динамики пригодны для кластеризации только после фильтрации по корректности времени и/или при использовании устойчивых преобразований.

4. Обнаружены логические несогласованности и выбросы в параметрах здания:

   * несогласованность этажей (`fire_floor > building_floors`) — 105 записей,
   * экстремальные значения `fire_floor` до 908 и `building_floors` до 93, что подтверждает наличие аномалий и необходимость правил отбраковки и/или маркировки.

5. Распределение прямого ущерба имеет тяжёлый правый хвост: медиана 12 735, 75%-квантиль 68 856, максимум 2.086 млрд. Логарифмирование `direct_damage_log = log1p(direct_damage)` существенно стабилизирует масштаб и повышает пригодность признака для сравнений и кластеризации.

6. По заполненным значениям ущерба наблюдается различие периодов: 2009–2020 характеризуется более высоким уровнем `direct_damage_log`, но с более высокой долей пропусков по ущербу. Следовательно, сравнение периодов корректно только при явном учёте неполной наблюдаемости.

7. Расстояние до пожарной части (`distance_to_station`) заполнено хорошо (пропуски ≈ 2.2%) и показывает структурные различия по типам населённых пунктов: для отдельных типов наблюдаются более высокие типичные расстояния и более широкий разброс, что согласуется с различиями инфраструктурной доступности.

## 4.2. Итоговые выводы по постановке признаков

8. Признаки разделены на три смысловые группы в соответствии с заданием:

   * управляемые входные (характеристики реагирования и тушения),
   * наблюдаемые входные (контекст объекта и территории),
   * выходные показатели (последствия и ущерб),
     а также сформированы комбинированные наборы признаков для экспериментов.

9. Из-за критической доли пропусков из экспериментов исключены признаки с заполненностью ниже порога 10% (пропуски > 90%), в частности `risk_category_code` и `fpo_class_code`, а из выходов — `fatalities` и `injuries`. Фактически «выходной» блок для экспериментов оказался представлен главным образом `direct_damage_log` и производными от него показателями тяжести на подвыборке заполненных значений.

10. Импутация использовалась как вычислительная процедура на этапе формирования матрицы признаков для конкретного эксперимента и не подменяла исходную структуру пропусков в `clean_df/analysis_df`. Это позволило одновременно сохранить корректный аудит качества данных и обеспечить воспроизводимость кластеризации при разных режимах обработки пропусков.

## 4.3. Выбранное решение и обоснование

11. Проведена серия экспериментов по кластеризации для групп признаков и их комбинаций при варьировании:

* алгоритма (KMeans и агломеративная кластеризация),
* числа кластеров (k = 2…10),
* режима обработки пропусков (`dropna` и `impute`),
  с оценкой качества по silhouette, Davies–Bouldin, Calinski–Harabasz и по метрикам согласованности с ранжированием тяжести (Spearman, purity, NMI). Для исключения «дегенеративных» разбиений вводились ограничения на минимальный размер кластера (≥ 30) и минимальную долю самого малого кластера (≥ 1%).

12. Для сопоставления кластеров с «линейкой рангов пожаров» сформирован прокси-показатель тяжести:
    [
    severity_score = 5 \cdot fatalities + 2 \cdot injuries + direct_damage_log
    ]
    и эталонный ранг `rank_ref`, задаваемый по квартилям `severity_score` на положительных значениях. В силу неполной заполненности потерь и ущерба ранжирование фактически отражает тяжесть на подвыборке заполненных значений и преимущественно определяется `direct_damage_log`.

13. Лучшим по совокупному критерию качества выбран вариант:

* `feature_set = managed`,
* `algorithm = kmeans`,
* `imputation = impute`,
* `k = 7`,
* `rows_used = 1700`,
* `final_score ≈ 0.372`.
  Он показывает умеренную разделимость кластеров (silhouette ≈ 0.297) и наиболее высокую среди рассмотренных вариантов согласованность с ранжированием тяжести (Spearman ≈ 0.238 при сопоставимых значениях purity и NMI).

14. Варианты на наблюдаемых признаках (например, `observed` при k=2) демонстрировали более высокий silhouette, однако хуже согласовывались с ранжированием тяжести, что снижало их приоритет при условии задания о близости к шкале рангов пожаров.

## 4.4. Интерпретация выбранного решения по экспортируемым меткам

15. Экспорт результата (`labels_best.csv`) содержит 1700 наблюдений, использованных в лучшем варианте, и включает:

* идентификатор записи `row_id`,
* метку кластера `cluster_best` (7 кластеров: 0–6),
* эталонный ранг тяжести `rank_ref` (значения 2–5).
  Распределение по рангам в этой подвыборке близко к равномерному, что соответствует построению `rank_ref` по квартилям.

16. По среднему `rank_ref` кластеры упорядочиваются от «более лёгких» к «более тяжёлым». Наблюдается монотонный рост средней тяжести при переходе от кластеров 0–1 к кластерам 4–6–5; наибольшую среднюю тяжесть демонстрирует кластер 5. Это подтверждает возможность трактовать результат как дискретные уровни тяжести в терминах характеристик реагирования и тушения, доступных в данных.

## 4.5. Итог

17. Требования задания выполнены: проведён EDA и контроль качества с маркировкой и исключением некорректных записей, признаки разделены на управляемые/наблюдаемые/выходные, выполнена кластеризация по множеству наборов признаков, реализован интерактивный интерфейс для EDA, кластеризации и сравнения вариантов, выбран и зафиксирован лучший вариант, выполнено ранжирование вариантов по степени согласованности с эталонным ранжированием тяжести.

18. Основным ограничением остаётся неполнота выходных показателей, из-за чего ранжирование тяжести устойчиво интерпретируется только на подвыборке заполненных значений. Для повышения достоверности дальнейших выводов целесообразно расширять заполненность выходов и усиливать валидацию временной логики с переходом к интервальным длительностям.



# 5. Приложение: параметры, версии, воспроизводимость
- SEED = 42, все алгоритмы используют единый seed.
- Версии библиотек выводятся в начале ноутбука.
- Пути и имена файлов данных фиксируются при загрузке.
- Артефакты сохраняются: clean_df.parquet (fallback CSV), analysis_df.parquet, experiments_results.csv, best_variant.json, labels_best.csv.


In [15]:

# Сохранение артефактов для воспроизводимости

def safe_save(df, parquet_path, csv_path):
    try:
        out = df.copy()
        obj_cols = out.select_dtypes(include=['object']).columns
        for c in obj_cols:
            out[c] = out[c].astype('string')
        out.to_parquet(parquet_path, index=False)
        print(f'Сохранено {parquet_path}')
    except Exception as e:
        print(f'Parquet недоступен ({e}), сохраняю CSV {csv_path}')
        df.to_csv(csv_path, index=False)

safe_save(clean_df, 'clean_df.parquet', 'clean_df.csv')
safe_save(analysis_df, 'analysis_df.parquet', 'analysis_df.csv')
if 'results_df' in globals() and not results_df.empty:
    results_df.to_csv('experiments_results.csv', index=False)
    print('Сохранено experiments_results.csv')
if Path('best_variant.json').exists():
    print('best_variant.json уже сохранён выше')


Сохранено clean_df.parquet
Сохранено analysis_df.parquet
Сохранено experiments_results.csv
best_variant.json уже сохранён выше
